In [1]:
#import libraries

%pip install pandas numpy
import pandas as pd
import numpy as np
import pyarrow as pa
import os

for ext_name in ['pandas.interval', 'pandas.period']:
    try:
        pa.unregister_extension_type(ext_name)
    except KeyError:
        pass

# Define file paths based on the required repository structure
TAXI_DATA_PATH = '../data/raw/tlc/yellow_tripdata_2023-01.parquet'
ZONE_LOOKUP_PATH = '../data/raw/lookup/taxi_zone_lookup.csv'
OUTPUT_CURATED_PATH = '../data/curated/part1_taxi_curated.parquet'


# Load the raw datasets
df = pd.read_parquet(TAXI_DATA_PATH, engine ='fastparquet')
df_zone = pd.read_csv(ZONE_LOOKUP_PATH)



Note: you may need to restart the kernel to use updated packages.


In [2]:
# Merge for Pickup Location details
df = df.merge(
    df_zone[['LocationID', 'Borough', 'Zone']], 
    left_on='PULocationID', 
    right_on='LocationID', 
    how='left'
).rename(columns={'Borough': 'PU_borough', 'Zone': 'PU_zone'}).drop(columns=['LocationID'])

# Merge for Dropoff Location details
df = df.merge(
    df_zone[['LocationID', 'Borough', 'Zone']], 
    left_on='DOLocationID', 
    right_on='LocationID', 
    how='left'
).rename(columns={'Borough': 'DO_borough', 'Zone': 'DO_zone'}).drop(columns=['LocationID'])

In [3]:
# --- B. Time fields ---
# Ensure datetime format
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

df['pickup_date'] = df['tpep_pickup_datetime'].dt.date
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df['weekday'] = df['tpep_pickup_datetime'].dt.day_name().str[:3] # Mon-Sun
df['week_of_year'] = df['tpep_pickup_datetime'].dt.isocalendar().week

# --- D. Engineered fields ---
# trip_duration_min
df['trip_duration_min'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60.0

# speed_mph (handle zero duration safely)
df['speed_mph'] = np.where(
    df['trip_duration_min'] > 0, 
    df['trip_distance'] / (df['trip_duration_min'] / 60.0), 
    0.0
)

# fare_per_mile (handle zero distance safely)
df['fare_per_mile'] = np.where(
    df['trip_distance'] > 0, 
    df['fare_amount'] / df['trip_distance'], 
    0.0
)

# tip_rate (handle zero total amount safely)
df['tip_rate'] = np.where(
    df['total_amount'] > 0, 
    df['tip_amount'] / df['total_amount'], 
    0.0
)

# distance_bucket with bins: [0,1), [1,3), [3,5), [5,10), [10,+)
bins = [0, 1, 3, 5, 10, np.inf]
labels = ['[0,1)', '[1,3)', '[3,5)', '[5,10)', '[10,+)']
df['distance_bucket'] = pd.cut(df['trip_distance'], bins=bins, labels=labels, right=False)

In [4]:
# Initialize tracker for the before/after table
cleaning_tracker = []
initial_count = len(df)

# Rule 1: Remove rows where pickup or dropoff timestamp is missing
df_clean = df.dropna(subset=['tpep_pickup_datetime', 'tpep_dropoff_datetime'])
count_r1 = len(df_clean)
cleaning_tracker.append({
    'Rule': '1. Missing pickup/dropoff timestamp', 
    'Rows Before': initial_count, 
    'Rows After': count_r1, 
    'Removed': initial_count - count_r1
})

# Rule 2: Remove rows where trip_duration_min <= 0
df_clean = df_clean[df_clean['trip_duration_min'] > 0]
count_r2 = len(df_clean)
cleaning_tracker.append({
    'Rule': '2. trip_duration_min <= 0', 
    'Rows Before': count_r1, 
    'Rows After': count_r2, 
    'Removed': count_r1 - count_r2
})

# Rule 3: Remove rows where trip_distance <= 0
df_clean = df_clean[df_clean['trip_distance'] > 0]
count_r3 = len(df_clean)
cleaning_tracker.append({
    'Rule': '3. trip_distance <= 0', 
    'Rows Before': count_r2, 
    'Rows After': count_r3, 
    'Removed': count_r2 - count_r3
})

# Rule 4: Remove rows where passenger_count is missing or <= 0
df_clean = df_clean[(df_clean['passenger_count'].notnull()) & (df_clean['passenger_count'] > 0)]
count_r4 = len(df_clean)
cleaning_tracker.append({
    'Rule': '4. passenger_count missing or <= 0', 
    'Rows Before': count_r3, 
    'Rows After': count_r4, 
    'Removed': count_r3 - count_r4
})

# Rule 5: Remove rows where total_amount <= 0
df_clean = df_clean[df_clean['total_amount'] > 0]
count_r5 = len(df_clean)
cleaning_tracker.append({
    'Rule': '5. total_amount <= 0', 
    'Rows Before': count_r4, 
    'Rows After': count_r5, 
    'Removed': count_r4 - count_r5
})

# Rule 6: Remove duplicates using the specified key
dup_key = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'total_amount']
df_clean = df_clean.drop_duplicates(subset=dup_key)
count_r6 = len(df_clean)
cleaning_tracker.append({
    'Rule': '6. Duplicates (by defined key)', 
    'Rows Before': count_r5, 
    'Rows After': count_r6, 
    'Removed': count_r5 - count_r6
})

# Print the before/after table
tracker_df = pd.DataFrame(cleaning_tracker)
print("Cleaning Rules Summary Table:")
display(tracker_df) # Use display() for a nicer formatted table in Jupyter

Cleaning Rules Summary Table:


,Rule,Rows Before,Rows After,Removed
0,1. Missing pickup/dropoff timestamp,3066766,3066766,0
1,2. trip_duration_min <= 0,3066766,3065645,1121
2,3. trip_distance <= 0,3065645,3020831,44814
3,4. passenger_count missing or <= 0,3020831,2906607,114224
4,5. total_amount <= 0,2906607,2884397,22210
5,6. Duplicates (by defined key),2884397,2884396,1


In [5]:
# Define the final list of required columns
final_columns = [
    # Geography
    'PULocationID', 'DOLocationID', 'PU_borough', 'PU_zone', 'DO_borough', 'DO_zone',
    # Time
    'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'pickup_date', 'pickup_hour', 'weekday', 'week_of_year',
    # Core numeric
    'passenger_count', 'trip_distance', 'fare_amount', 'tip_amount', 'total_amount', 'payment_type',
    # Engineered
    'trip_duration_min', 'speed_mph', 'fare_per_mile', 'tip_rate', 'distance_bucket'
]


if not isinstance(df_clean, pd.DataFrame):
    raise TypeError(f"df_clean is {type(df_clean)}, expected pandas DataFrame")

# Select final columns
df_final = df_clean[final_columns].copy()

# Ensure the output directory exists
os.makedirs(os.path.dirname(OUTPUT_CURATED_PATH), exist_ok=True)

#for col in df_final.columns:
    #if str(df_final[col].dtype).startswith("period"):
        #df_final[col] = df_final[col].astype(str)

# Save to parquet
df_final.to_parquet(OUTPUT_CURATED_PATH, index=False, engine='pyarrow')
print(f"Curated dataset successfully saved to: {OUTPUT_CURATED_PATH}")

Curated dataset successfully saved to: ../data/curated/part1_taxi_curated.parquet


In [7]:
df = pd.read_parquet(OUTPUT_CURATED_PATH)
df.head()

,PULocationID,DOLocationID,PU_borough,PU_zone,DO_borough,DO_zone,tpep_pickup_datetime,tpep_dropoff_datetime,pickup_date,pickup_hour,...,trip_distance,fare_amount,tip_amount,total_amount,payment_type,trip_duration_min,speed_mph,fare_per_mile,tip_rate,distance_bucket
0,161,141,Manhattan,Midtown Center,Manhattan,Lenox Hill West,2023-01-01 00:32:10,2023-01-01 00:40:36,2023-01-01,0,...,0.97,9.3,0.00,14.30,2,8.433333,6.901186,9.587629,0.000000,"[0,1)"
1,43,237,Manhattan,Central Park,Manhattan,Upper East Side South,2023-01-01 00:55:08,2023-01-01 01:01:27,2023-01-01,0,...,1.10,7.9,4.00,16.90,1,6.316667,10.448549,7.181818,0.236686,"[1,3)"
2,48,238,Manhattan,Clinton East,Manhattan,Upper West Side North,2023-01-01 00:25:04,2023-01-01 00:37:49,2023-01-01,0,...,2.51,14.9,15.00,34.90,1,12.750000,11.811765,5.936255,0.429799,"[1,3)"
3,107,79,Manhattan,Gramercy,Manhattan,East Village,2023-01-01 00:10:29,2023-01-01 00:21:19,2023-01-01,0,...,1.43,11.4,3.28,19.68,1,10.833333,7.920000,7.972028,0.166667,"[1,3)"
4,161,137,Manhattan,Midtown Center,Manhattan,Kips Bay,2023-01-01 00:50:34,2023-01-01 01:02:52,2023-01-01,0,...,1.84,12.8,10.00,27.80,1,12.300000,8.975610,6.956522,0.359712,"[1,3)"
